In [6]:
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path

In [7]:
PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT / "models"

PYTORCH_MODEL_PATH = MODEL_DIR / "mobilenetv2_best.pth"
ONNX_MODEL_PATH = MODEL_DIR / "mobilenetv2.onnx"

In [8]:
device = torch.device("cpu")  # ONNX export is safer on CPU

model = models.mobilenet_v2(pretrained=False)

model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, 2)
)

model.load_state_dict(torch.load(PYTORCH_MODEL_PATH, map_location=device))
model.eval()

print("Model loaded.")

c:\Users\cadg0\Downloads\CSC173-DeepCV-Gumisad\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\cadg0\Downloads\CSC173-DeepCV-Gumisad\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model loaded.


In [9]:
dummy_input = torch.randn(1, 3, 224, 224)

In [10]:
torch.onnx.export(
    model,
    dummy_input,
    ONNX_MODEL_PATH,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    },
    opset_version=11
)

print("ONNX model exported at:", ONNX_MODEL_PATH)

C:\Users\cadg0\AppData\Local\Temp\ipykernel_1188\1428794824.py:1: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0601 01:07:15.483000 1188 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\cadg0\AppData\Local\Programs\Python\Python314\Lib\copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).


[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\cadg0\Downloads\CSC173-DeepCV-Gumisad\venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "c:\Users\cadg0\Downloads\CSC173-DeepCV-Gumisad\venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "c:\Users\cadg0\Downloads\CSC173-DeepCV-Gumisad\venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\cadg0\Downloads\CSC173-DeepCV-Gumisad\venv\Lib\site

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX model exported at: c:\Users\cadg0\Downloads\CSC173-DeepCV-Gumisad\models\mobilenetv2.onnx


In [11]:
import os

print("File exists:", os.path.exists(ONNX_MODEL_PATH))
print("Size (MB):", os.path.getsize(ONNX_MODEL_PATH) / (1024*1024))

File exists: True
Size (MB): 0.2565488815307617


In [12]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(str(ONNX_MODEL_PATH))

input_name = session.get_inputs()[0].name

test_input = np.random.randn(1,3,224,224).astype(np.float32)

outputs = session.run(None, {input_name: test_input})

print("ONNX Output:", outputs)

ONNX Output: [array([[-0.55228686,  1.0084049 ]], dtype=float32)]
